# Project Introduction: CAPM, Fama-French, and Carhart

This notebook is designed as an applied finance and machine learning exercise focused on comparative asset-pricing models. The central question is simple but important: after controlling for market risk, do additional risk factors provide meaningful explanatory power for portfolio returns, or are they mostly redundant?

The analysis compares four nested linear factor models:

1. CAPM
2. Fama-French three-factor model
3. Carhart four-factor model
4. Fama-French five-factor model

These models are all linear regressions in which the dependent variable is the excess return of an asset or portfolio over the risk-free rate.

## 1. CAPM

The Capital Asset Pricing Model (CAPM) is the simplest benchmark. It assumes that the only systematic risk that should be rewarded is exposure to the market portfolio:

$$
R_{i,t} - R_{f,t} = \alpha_i + \beta_i\left(R_{m,t} - R_{f,t}\right) + \epsilon_{i,t}
$$

Where:
- $R_{i,t}$ is the return of asset $i$ at time $t$
- $R_{f,t}$ is the risk-free rate
- $R_{m,t}$ is the market return
- $\beta_i$ measures sensitivity to market risk
- $\alpha_i$ is the abnormal return not explained by market exposure
- $\epsilon_{i,t}$ is the regression error

The CAPM is historically grounded in the work of Sharpe (1964), Lintner (1965), and Mossin (1966). It remains a useful baseline because it provides a parsimonious explanation of expected returns based on a single systematic factor.

Academic references:
- Sharpe, W. F. (1964). Capital Asset Prices: A Theory of Market Equilibrium under Conditions of Risk. Journal of Finance.
- Lintner, J. (1965). The Valuation of Risk Assets and the Selection of Risky Investments in Stock Portfolios and Capital Budgets.
- Mossin, J. (1966). Equilibrium in a Capital Asset Market.

## 2. Fama-French Three-Factor Model

The Fama-French three-factor model extends CAPM by adding size and value risk factors. It is written as:

$$
R_{i,t} - R_{f,t} = \alpha_i + \beta_{i,m}\left(R_{m,t} - R_{f,t}\right) + s_i\,SMB_t + h_i\,HML_t + \epsilon_{i,t}
$$

Where:
- $SMB_t$ captures the performance difference between small and large firms
- $HML_t$ captures the performance difference between high and low book-to-market firms

This model is motivated by Fama and French (1993), who showed that size and value characteristics help explain cross-sectional variation in returns beyond market beta alone.

Academic reference:
- Fama, E. F., & French, K. R. (1993). Common Risk Factors in the Returns on Stocks and Bonds. Journal of Financial Economics.

## 3. Carhart Four-Factor Model

The Carhart model adds a momentum factor to the Fama-French framework:

$$
R_{i,t} - R_{f,t} = \alpha_i + \beta_{i,m}\left(R_{m,t} - R_{f,t}\right) + s_i\,SMB_t + h_i\,HML_t + m_i\,MOM_t + \epsilon_{i,t}
$$

Where:
- $MOM_t$ captures the momentum effect, i.e. the tendency of assets with strong recent performance to continue performing well in the short run

This extension is grounded in Carhart (1997), who showed that momentum is an important source of return persistence in mutual fund performance and related asset-pricing contexts.

Academic reference:
- Carhart, M. M. (1997). On Persistence in Mutual Fund Performance. Journal of Finance.

## 4. Fama-French Five-Factor Model

The Fama-French five-factor model further extends the framework by adding profitability and investment factors. The model is:

$$
R_{i,t} - R_{f,t} = \alpha_i + \beta_{i,m}\left(R_{m,t} - R_{f,t}\right) + s_i\,SMB_t + h_i\,HML_t + r_i\,RMW_t + c_i\,CMA_t + \epsilon_{i,t}
$$

Where:
- $RMW_t$ captures the difference between firms with robust and weak profitability
- $CMA_t$ captures the difference between firms with conservative and aggressive investment policies

This extension is grounded in Fama and French (2015), who introduced profitability and investment patterns as additional dimensions of systematic risk.

Academic reference:
- Fama, E. F., & French, K. R. (2015). A Five-Factor Asset Pricing Model. Journal of Financial Economics.

## Why this comparison matters

The comparison is useful because it allows us to test whether additional factors are statistically and economically meaningful. In practice, we will look at:
- the significance of each coefficient via p-values
- the overall explanatory power of the model via $R^2$ and adjusted $R^2$
- the joint significance of the regression using an F-test
- whether the added factors materially improve model fit relative to simpler benchmarks

In other words, this project is not only about fitting regressions. It is about asking a deeper financial and econometric question: do richer models capture genuine risk structure, or do they simply overfit the data?


### Imports & Libraries

In [2]:
import pandas as pd
import numpy as np

import yfinance as yf
import pandas_datareader.data as web

import plotly.graph_objects as go
from plotly.subplots import make_subplots
import plotly.express as px

from sklearn.linear_model import LinearRegression
from scipy import stats
import statsmodels.api as sm

## Risk factors (Fama-French 5 + Momentum)

Monthly returns for the 6 risk factors (2016–2026) retrieved via `pandas_datareader`. These factors 
will serve as explanatory variables in the regressions below: `MKT-RF` (market risk premium), `SMB` 
and `HML` (size and value from the 3-factor model), `RMW` and `CMA` (profitability and investment, 
5-factor extension), and `MOM` (momentum, added separately from the Carhart model).

We observe volatility spikes consistent with known market shocks (Covid in 2020, monetary tightening 
in 2022), which validates the data quality before using it in regression.

In [3]:
def fetch_fama_french_factors(start_date, end_date):

    ff5 = web.DataReader('F-F_Research_Data_5_Factors_2x3', 'famafrench', start_date, end_date)[0]
    momentum = web.DataReader('F-F_Momentum_Factor', 'famafrench', start_date, end_date)[0]
    
    factors = ff5.join(momentum)
    
    factors.rename(columns={'Mkt-RF': 'MKT-RF', 'Mom': 'MOM'}, inplace=True)
    factors.index = factors.index.to_timestamp(how='end').normalize()
    
    return factors


def plot_factors_history(factors_df):

    features = ['MKT-RF', 'SMB', 'HML', 'RMW', 'CMA', 'MOM']

    fig = make_subplots(
        rows=3, cols=2, 
        subplot_titles=features,
        vertical_spacing=0.1,  
        horizontal_spacing=0.05 
    )

    colors = ['cyan', 'lime', 'orange', 'magenta', 'yellow', 'pink']

    for i, feature in enumerate(features):
        row = (i // 2) + 1 
        col = (i % 2) + 1  

        fig.add_trace(
            go.Scatter(
                x=factors_df.index, 
                y=factors_df[feature], 
                mode='lines',
                line=dict(color=colors[i], width=1.5),
                name=feature
            ),
            row=row, col=col
        )
    
        fig.add_hline(
            y=0, 
            line_dash="dot", 
            line_color="white", 
            opacity=0.3, 
            row=row, col=col
        )
        
        fig.update_yaxes(title_text="Return (%)", row=row, col=col, title_font=dict(size=10))

    fig.update_layout(
        title={
            'text': "Factor History (Fama-French 5 + Momentum)",
            'y': 0.98,
            'x': 0.5,
            'xanchor': 'center',
            'yanchor': 'top',
            'font': dict(size=18, color='white')
        },
        template="plotly_dark",
        height=800, 
        width=1000,
        showlegend=False,
        hovermode="x unified"
    )

    fig.show()


start_date = '2016-01-01'
end_date = '2026-06-30'

factors = fetch_fama_french_factors(start_date, end_date)
    
print("Preview of factors (FF5 + Momentum):")
display(factors.head())

plot_factors_history(factors)

Preview of factors (FF5 + Momentum):


,MKT-RF,SMB,HML,RMW,CMA,RF,MOM
Date,,,,,,,
2016-01-31,-5.74,-3.44,2.08,2.78,3.05,0.01,1.49
2016-02-29,-0.07,0.87,-0.61,3.31,1.94,0.02,-4.34
2016-03-31,6.95,1.00,1.22,0.68,0.01,0.02,-5.04
2016-04-30,0.91,1.23,3.22,-2.85,1.77,0.01,-5.99
2016-05-31,1.78,-0.62,-1.62,-1.10,-2.54,0.01,1.45


## Building the sector portfolios

Six sub-portfolios built by style/sector (Growth, Value, Small_Mid, Defensive, Quality, Cyclical), each 
made up of 4 to 6 representative tickers, plus an equal-weighted total portfolio. Returns are computed 
as monthly changes in closing prices over the same period as the factors.

The goal is to get differentiated style exposures: each sub-portfolio should, in theory, react more or 
less strongly to specific factors (e.g. Growth vs MKT-RF and MOM, Value vs HML, Small_Mid vs SMB), which 
we'll test in the next section.

- Growth: ['AAPL', 'MSFT', 'NVDA', 'META', 'GOOGL', 'AMZN']
- Value: ['JPM', 'WFC', 'XOM', 'CVX', 'F', 'VZ']
- Small_Mid: ['MHK', 'ALK', 'HAS', 'ZION', 'ETSY']
- Defensif: ['PG', 'KO', 'WMT', 'DUK', 'SO']
- Qualite: ['JNJ', 'COST', 'ADP', 'MSCI']
- Cyclique: ['CAT', 'DE', 'FDX', 'MMM']

In [4]:

def fetch_portfolio_returns(portfolios_dict, start_date, end_date):

    all_tickers = []
    for tickers in portfolios_dict.values():
        all_tickers.extend(tickers)
    
    prices = yf.download(all_tickers, start=start_date, end=end_date, interval='1mo')['Close']
    
    equity_returns = prices.pct_change().dropna() * 100

    sub_portfolio_returns = pd.DataFrame(index=equity_returns.index)
    for name, tickers in portfolios_dict.items():
        sub_portfolio_returns[name] = equity_returns[tickers].mean(axis=1)

    total_portfolio_return = pd.DataFrame(index=equity_returns.index)
    total_portfolio_return['Total_Portfolio'] = equity_returns.mean(axis=1)

    all_returns = pd.concat([
        equity_returns,           
        sub_portfolio_returns,    
        total_portfolio_return    
    ], axis=1)

    all_returns.index = all_returns.index.to_period('M').to_timestamp(how='end').normalize()

    return all_returns

def plot_portfolio_returns(returns_df, portfolios_to_plot):

    fig = make_subplots(
        rows=4, cols=2, 
        subplot_titles=portfolios_to_plot,
        vertical_spacing=0.08,   
        horizontal_spacing=0.05
    )
    
    colors = ['cyan', 'lime', 'orange', 'magenta', 'yellow', 'pink', 'white']

    for i, port in enumerate(portfolios_to_plot):
        row = (i // 2) + 1  
        col = (i % 2) + 1   
        
        fig.add_trace(
            go.Scatter(
                x=returns_df.index, 
                y=returns_df[port], 
                mode='lines',
                line=dict(color=colors[i % len(colors)], width=1.5),
                name=port
            ),
            row=row, col=col
        )
        
        fig.add_hline(
            y=0, 
            line_dash="dot", 
            line_color="rgba(255, 255, 255, 0.3)", 
            row=row, col=col
        )
        
        fig.update_yaxes(title_text="Return (%)", row=row, col=col, title_font=dict(size=10))

    fig.update_layout(
        title={
            'text': "Portfolio Performance History (2016 - 2026)",
            'y': 0.98,
            'x': 0.5,
            'xanchor': 'center',
            'yanchor': 'top',
            'font': dict(size=18, color='white')
        },
        template="plotly_dark",
        height=1000,  
        showlegend=False,
        hovermode="x unified"
    )

    fig.show()


portfolio = {
    'Growth': ['AAPL', 'MSFT', 'NVDA', 'META', 'GOOGL', 'AMZN'],
    'Value': ['JPM', 'WFC', 'XOM', 'CVX', 'F', 'VZ'],
    'Small_Mid': ['MHK', 'ALK', 'HAS', 'ZION', 'ETSY'],
    'Defensif': ['PG', 'KO', 'WMT', 'DUK', 'SO'],
    'Qualite': ['JNJ', 'COST', 'ADP', 'MSCI'],
    'Cyclique': ['CAT', 'DE', 'FDX', 'MMM']
    }
    
portfolio_to_plot = ['Growth', 'Value', 'Small_Mid', 'Defensif', 'Qualite', 'Cyclique', 'Total_Portfolio']

all_returns = fetch_portfolio_returns(portfolio, start_date, end_date)

plot_portfolio_returns(all_returns, portfolio_to_plot)
display(all_returns.head())


[*********************100%***********************]  30 of 30 completed


,AAPL,ADP,ALK,AMZN,CAT,COST,CVX,DE,DUK,ETSY,...,WMT,XOM,ZION,Growth,Value,Small_Mid,Defensif,Qualite,Cyclique,Total_Portfolio
Date,,,,,,,,,,,,,,,,,,,,,
2016-02-29,-0.128762,1.925613,5.372291,-5.873938,8.772489,-0.452074,-2.280152,4.116360,-0.312631,2.319585,...,-0.030183,3.888829,-5.727015,-2.672485,-0.556916,2.421949,-0.396199,1.427743,5.134282,0.566682
2016-03-31,12.721054,6.588310,10.987799,7.442264,13.057595,5.032293,14.333631,-3.254508,8.616046,9.571785,...,4.005203,4.291939,13.555363,9.235540,6.899728,9.181027,6.180520,4.877707,8.778083,7.608083
2016-04-30,-13.992084,-1.415723,-14.130688,11.109429,2.531809,-5.718945,7.106964,9.247969,-2.354981,0.919539,...,-2.365325,5.754311,13.672059,-2.838368,3.410388,1.531406,-2.632403,-0.259491,3.424662,0.352927
2016-05-31,7.177268,-0.678409,-5.323831,9.581709,-6.703580,0.432064,-0.109943,-2.163869,0.343005,5.353078,...,6.622102,1.547648,2.050213,10.408069,1.062667,1.463604,1.492032,1.596294,-1.930256,2.742225
2016-06-30,-4.265982,5.229954,-12.213858,-0.991993,4.551120,5.558902,3.792048,-0.780166,9.663830,3.675677,...,3.164768,5.302228,-10.314063,-2.992477,0.083423,-5.231213,5.639794,3.770742,0.011377,-0.009432


## Excess returns and factor sensitivity

Portfolio returns are joined with the factors and converted into excess returns (portfolio return minus 
the risk-free rate `RF`), a necessary step for a properly specified factor regression (CAPM / Fama-French 
style).

The scatter plot of `Total_Portfolio_Excess` vs. factors shows a clearly linear relationship with 
`MKT-RF`: this is expected, since the total portfolio is diversified and therefore dominated by its 
market beta. The relationship with the other factors (SMB, HML, RMW, CMA, MOM) is more diffuse, which 
makes sense: on an aggregated portfolio, the individual style exposures of the sub-portfolios tend to 
cancel each other out.

This linearity becomes noticeably stronger when the analysis is run on a targeted sub-portfolio instead 
of the total portfolio — for example `Value_Excess` vs. `HML`, or `Small_Mid_Excess` vs. `SMB` — because 
it isolates the exposure to the factor that sub-portfolio was actually built around, rather than diluting 
it into an average. We'll verify this in the next regression by changing `selected_portfolio`.

In [5]:

def prepare_regression_data(returns, factors):
    final_data = returns.join(factors, how='inner').dropna()
    for portfolio_name in portfolio_to_plot:
        
        excess_col = f'{portfolio_name}_Excess'
        final_data[excess_col] = final_data[portfolio_name] - final_data['RF']
        
    return final_data


def plot_factor_sensitivities(data: pd.DataFrame, y_target: str, features: list):
    fig = make_subplots(
        rows=3, cols=2, 
        subplot_titles=[f"Sensibilité au facteur {feat}" for feat in features],
        vertical_spacing=0.12,
        horizontal_spacing=0.08
    )

    colors = ['cyan', 'lime', 'orange', 'magenta', 'yellow', 'pink']

    for i, feature in enumerate(features):
        row = (i // 2) + 1  
        col = (i % 2) + 1 
        
        x_val = data[feature]
        y_val = data[y_target]
        
        fig.add_trace(
            go.Scatter(
                x=x_val, 
                y=y_val, 
                mode='markers',
                marker=dict(color=colors[i % len(colors)], size=6, opacity=0.5),
                name=f"{feature} scatter"
            ),
            row=row, col=col
        )

        fig.update_xaxes(title_text=f"Rendement du facteur {feature} (%)", row=row, col=col, title_font=dict(size=10))
        fig.update_yaxes(title_text=f"{y_target} (%)", row=row, col=col, title_font=dict(size=10))

    fig.update_layout(
        title={
            'text': f"Analyse de Linéarité : {y_target} vs Facteurs de Risque",
            'y': 0.98,
            'x': 0.5,
            'xanchor': 'center',
            'yanchor': 'top',
            'font': dict(size=18, color='white')
        },
        template="plotly_dark",
        height=900,
        width=1000,
        showlegend=False,
        hovermode="closest"
    )

    fig.show()

df_final = prepare_regression_data(all_returns, factors)

selected_portfolio = 'Total_Portfolio' # Growth, Value, Small_Mid, Defensif, Qualite, Cyclique, Total_Portfolio
features = ['MKT-RF', 'SMB', 'HML', 'RMW', 'CMA', 'MOM']

target_y_var = f'{selected_portfolio}_Excess'
plot_factor_sensitivities(df_final, target_y_var, features)

## 1. CAPM regression by portfolio

### What the regression tests

For each portfolio, we estimate a single-factor model by OLS (ordinary least squares):

$$R_{i,t} - R_{f,t} = \alpha_i + \beta_i (R_{m,t} - R_{f,t}) + \varepsilon_{i,t}$$

where $R_i - R_f$ is the excess return of portfolio $i$ and $R_m - R_f$ is the market excess return 
(`MKT-RF`). Two implementations are compared (a "manual" calculation via `LinearRegression` + matrix formulas, 
and `statsmodels.OLS`) — they give identical results, which validates the manual computation of standard 
errors and test statistics.

What each statistic tells us:

- **Beta**: the portfolio's sensitivity to the market. Beta > 1 amplifies market moves (aggressive portfolio), 
  beta < 1 dampens them (defensive portfolio). It's the only systematic risk factor recognized by the CAPM, 
  so in theory the only one that should be rewarded.
- **Alpha**: excess return *not explained* by the market. Under pure CAPM theory (efficient market), alpha 
  should be zero in expectation. A significant positive alpha suggests either an inefficiency (arbitrage 
  opportunity) or — more plausibly here — a risk premium compensating for a factor omitted from the model 
  (size, style, quality, momentum, etc.).
- **t-stat / p-value**: individual significance test for each coefficient (H₀: coefficient = 0). With ~120 
  monthly observations, a coefficient is considered significant if p-value < 0.05.
- **F-stat / p-value (F)**: overall significance test for the model (H₀: the model explains nothing, R² = 0). 
  With a single regressor here, this test is redundant with the beta's t-stat (F = t²), but it will become 
  more informative in the multi-factor regressions ahead.
- **R²**: share of return variance explained by the market alone. The residual (1 - R²) corresponds to 
  *idiosyncratic* risk specific to the portfolio, not explained by market exposure.

### Reading the results

| Portfolio | Beta | Alpha (monthly) | p_alpha | R² |
|---|---|---|---|---|
| Total_Portfolio | 0.96 | 0.28% | 0.033 | **0.905** |
| Growth | 1.22 | 1.12% | 0.005 | 0.635 |
| Qualite | 0.75 | 0.47–0.65% | 0.07–0.01 | 0.60 |
| Cyclique | 1.09 | 0.27% | 0.52 (n.s.) | 0.55 |
| Small_Mid | 1.33 | -0.52% | 0.29 (n.s.) | 0.56 |
| Value | 0.93 | -0.09% | 0.80 (n.s.) | 0.53 |
| Defensif | 0.38 | 0.41% | 0.20 (n.s.) | **0.21** |

All betas are highly significant (p < 0.001), which is expected: market exposure is rarely zero for an equity 
portfolio. The ranking of betas is consistent with economic intuition:

- **Small_Mid (1.33) and Growth (1.22)** are the most aggressive — consistent with growth/small-cap names, 
  historically more volatile and more cyclically sensitive.
- **Defensif (0.38)** has by far the lowest beta — consistent with inelastic-demand sectors (staples, 
  healthcare, utilities), weakly correlated with the economic cycle.
- **Total_Portfolio (0.96)**, close to 1, confirms that diversification pulls the aggregated portfolio back 
  toward the market's profile — logical since it's an average of the 6 styles.

On alpha, only **Growth** (1.12%, p = 0.005) and **Total_Portfolio** (0.28%, p = 0.033) are significant at 
the 5% level. All other alphas are not statistically different from zero: we cannot reject the CAPM hypothesis 
for these portfolios taken individually.

### Economic interpretation

1. **The CAPM "works" at the aggregate level**, in the sense that market beta is the main driver of returns — 
   the total portfolio's R² (0.905) shows that 90% of its variance is explained by the market factor alone. 
   This is consistent with theory: a diversified portfolio eliminates idiosyncratic risk and retains only 
   systematic risk.

2. **But the CAPM is incomplete at the style level.** R² drops sharply once we move to the sub-portfolios 
   (0.21 to 0.64), especially for Defensif. This means a large share (up to ~80% for Defensif) of the variance 
   in these returns is *not* explained by the market alone — there's an additional risk factor to identify 
   (exactly the role of SMB, HML, RMW, CMA, and MOM in the next section).

3. **The significant alpha for Growth (+1.12%/month) is the most interesting result to dig into.** Two 
   readings are possible: (a) the Growth portfolio captures a non-market risk premium (typically linked to 
   momentum or an unmodeled quality/growth factor), or (b) it's simply an artifact of the estimation window 
   (2016–2026 was structurally favorable to US tech/growth names, e.g. AAPL, MSFT, NVDA, META). A genuinely 
   persistent economic alpha is rare and suspicious at this level — caution is warranted before treating this 
   as an active-management signal rather than a sample bias.

4. **The weak explanatory power for Defensif (R² = 0.21)** suggests this style is driven by forces other than 
   the overall market cycle — interest rates, risk aversion, sector rotation. It's a natural candidate for an 
   RMW-type factor (profitability/quality) or a macro variable (rates) absent from the current model.



### 1a. Manual CAPM Implementation (matrix algebra)

**What the code does.** `capm_ols()` re-implements single-factor OLS "from scratch": it fits $y = \alpha + \beta x$ with `sklearn.LinearRegression`, then manually reconstructs the design matrix $X = [\mathbf{1}, x]$ and computes:

- the residual sum of squares $RSS = \sum e_i^2$ and residual variance $\hat\sigma^2 = RSS/(n-k-1)$,
- the coefficient covariance matrix $\hat\sigma^2 (X^\top X)^{-1}$, whose diagonal gives the standard errors,
- $t$-statistics ($\hat\beta / SE$) and their two-sided $p$-values from the Student-$t$ distribution,
- the $F$-statistic and $R^2$ from the explained/residual sum-of-squares decomposition ($TSS = ESS + RSS$).

The function is first called once directly on `Total_Portfolio` (the **raw** return column, not yet converted to excess return), then in a loop over every `_Excess` column to build a summary table sorted by $R^2$.

**Why this matters on the ML side.** This is the textbook closed-form OLS solution $\hat\beta = (X^\top X)^{-1}X^\top y$, exactly what `sklearn` and `statsmodels` compute under the hood — reproducing it manually is a good sanity check that the "black-box" regressions used later in the notebook (Section 1b onward) aren't hiding anything. Note also a subtle and instructive detail: the first standalone call uses `Total_Portfolio` (raw return) rather than `Total_Portfolio_Excess`, so its alpha (0.47%) is mechanically larger than the one obtained inside the loop for the excess-return version (0.29%) — the gap (~0.18%) is essentially the average monthly risk-free rate over the sample, a reminder of why CAPM must be estimated on *excess* returns and not raw returns.

**Results obtained.** Ranked by $R^2$: `Total_Portfolio` (β=0.96, R²=0.905), `Growth` (β=1.23, R²=0.629), `Qualite` (β=0.75, R²=0.600), `Small_Mid` (β=1.32, R²=0.548), `Cyclique` (β=1.08, R²=0.532), `Value` (β=0.93, R²=0.528), `Defensif` (β=0.38, R²=0.205). These numbers match the manual "Reading the results" table given in Section 1 above and confirm the market-beta ranking discussed there (Small_Mid/Growth aggressive, Defensif defensive).

In [6]:
def capm_ols(data, portfolio_col, market_factor='MKT-RF'):

    y = data[portfolio_col].values
    X = data[[market_factor]].values
    n = X.shape[0]
    k = 1
    
    model = LinearRegression()
    model.fit(X, y)
    
    alpha = model.intercept_
    beta = model.coef_[0]
    residuals = y - model.predict(X)
    
    X_design = np.column_stack([np.ones(n), X])
    coefs = np.array([alpha, beta])
    
    RSS = np.sum(residuals**2)
    dof = n - k - 1
    sigma2 = RSS / dof
    
    XtX_inv = np.linalg.inv(X_design.T @ X_design)
    se_coefs = np.sqrt(sigma2 * np.diag(XtX_inv))
    
    t_stats = coefs / se_coefs
    p_values_t = 2 * (1 - stats.t.cdf(np.abs(t_stats), df=dof))
    
    results_df = pd.DataFrame({
        'Coefficient': coefs,
        'Std Error': se_coefs,
        't-stat': t_stats,
        'p-value': p_values_t
    }, index=['Alpha', f'Beta ({market_factor})'])
    
    TSS = np.sum((y - np.mean(y))**2)
    ESS = TSS - RSS
    
    F_stat = (ESS / k) / (RSS / dof)
    p_value_F = 1 - stats.f.cdf(F_stat, k, dof)
    r_squared = 1 - RSS / TSS
    
    print(f"\n--- ANALYSE CAPM : {portfolio_col} ---")
    print(results_df.round(4))
    print("-" * 45)
    print(f"F-statistique : {F_stat:.4f}")
    print(f"p-value (F)   : {p_value_F:.6f}")
    print(f"R²            : {r_squared:.4f}")
    print("=" * 45)
    
    
    return {
        'alpha': alpha,
        'beta': beta,
        'p_alpha': p_values_t[0],
        'p_beta': p_values_t[1],
        'r2': r_squared,
        'f_stat': F_stat,
        'p_f': p_value_F
    }


capm_ols(df_final, selected_portfolio) 

summary_rows = []
for port in portfolio_to_plot:
    col_name = 'Total_Portfolio_Excess' if port == 'Total_Portfolio' else f'{port}_Excess'
    
    if col_name in df_final.columns:
        res = capm_ols(df_final, col_name) 
        
        summary_rows.append({
            'Portfolio': port,
            'Alpha': round(res['alpha'], 4),
            'Beta': round(res['beta'], 4),
            'p_alpha': round(res['p_alpha'], 4),
            'p_beta': round(res['p_beta'], 4),
            'R2': round(res['r2'], 6),
            'p_F': round(res['p_f'], 6),
        })

summary = pd.DataFrame(summary_rows)
print("\n=== Overall Summary ===")
display(summary.sort_values('R2', ascending=False))


--- ANALYSE CAPM : Total_Portfolio ---
               Coefficient  Std Error   t-stat  p-value
Alpha               0.4722     0.1297   3.6408   0.0004
Beta (MKT-RF)       0.9580     0.0278  34.4201   0.0000
---------------------------------------------
F-statistique : 1184.7446
p-value (F)   : 0.000000
R²            : 0.9059

--- ANALYSE CAPM : Growth_Excess ---
               Coefficient  Std Error   t-stat  p-value
Alpha               1.0267     0.3968   2.5875   0.0108
Beta (MKT-RF)       1.2284     0.0851  14.4281   0.0000
---------------------------------------------
F-statistique : 208.1688
p-value (F)   : 0.000000
R²            : 0.6286

--- ANALYSE CAPM : Value_Excess ---
               Coefficient  Std Error   t-stat  p-value
Alpha              -0.1314     0.3701  -0.3551   0.7231
Beta (MKT-RF)       0.9323     0.0794  11.7409   0.0000
---------------------------------------------
F-statistique : 137.8487
p-value (F)   : 0.000000
R²            : 0.5285

--- ANALYSE CAPM : Sma

,Portfolio,Alpha,Beta,p_alpha,p_beta,R2,p_F
6,Total_Portfolio,0.2902,0.9588,0.0280,0.0,0.905053,0.0
0,Growth,1.0267,1.2284,0.0108,0.0,0.628588,0.0
4,Qualite,0.4759,0.7487,0.0663,0.0,0.599945,0.0
2,Small_Mid,-0.4292,1.3162,0.3942,0.0,0.548389,0.0
5,Cyclique,0.3660,1.0815,0.3918,0.0,0.532451,0.0
1,Value,-0.1314,0.9323,0.7231,0.0,0.528462,0.0
3,Defensif,0.4222,0.3794,0.1811,0.0,0.205018,0.0


### 1b. CAPM via `statsmodels` + Diagnostic Dashboard

**What the code does.** `run_capm_analysis()` re-estimates the same single-factor CAPM regression, this time using `statsmodels.OLS`, which returns the full inferential toolkit (coefficients, standard errors, $R^2$/adjusted $R^2$, $F$-statistic, fitted values, residuals) without needing to hand-code the linear algebra. `plot_capm_visualizations()` then renders a 6-panel diagnostic dashboard for the currently selected portfolio (`selected_portfolio`, set to `Total_Portfolio`):

1. Scatter of excess return vs. `MKT-RF` with the fitted CAPM line.
2. Residuals vs. fitted values — used to check for **heteroscedasticity** (a funnel/fan shape would indicate the constant-variance assumption behind classical standard errors is violated).
3. Histogram of residuals — a rough visual normality check that underpins the validity of the $t$/$F$ tests in a finite sample.
4. Forest plot of alpha and beta with their 95% confidence intervals.
5. Observed vs. predicted returns, benchmarked against the $y=x$ line (a well-fit model should hug this diagonal).
6. Time series overlay of observed vs. fitted excess returns, useful for spotting periods where the model systematically over/under-predicts (e.g. around regime changes like the 2020 Covid shock or the 2022 rate-hiking cycle).

**Why this matters on the ML side.** This cell shifts the notebook from "compute a coefficient" to "diagnose whether the linear-regression assumptions hold" — the same checks (residual patterns, normality, leverage) that apply to any linear model, not just CAPM. Getting identical point estimates here as in the manual implementation (Section 1a) is itself a validation step: it confirms the hand-rolled formulas were implemented correctly.

**Results obtained (Total_Portfolio).** α = 0.290 (p = 0.028), β = 0.959, R² = 0.905, adjusted R² = 0.904 — consistent to the fourth decimal with the manual computation in Section 1a. The residual plots (panels 2–3) show no obvious funnel shape or strong skew, giving no immediate red flag on homoscedasticity or normality, though this is only visual — the "Limitations" discussion in Section 1 recommends formal tests (Breusch-Pagan, Jarque-Bera) as a next step.

In [ ]:
def run_capm_analysis(data, portfolio_col, market_factor: str = 'MKT-RF'):
    
    y = data[portfolio_col]
    X = sm.add_constant(data[market_factor])

    model = sm.OLS(y, X).fit()

    summary_table = model.summary2().tables[1]
    summary_table.index = ['Alpha', 'Beta (MKT-RF)']
    summary_table = summary_table.rename(columns={'P>|t|': 'p-value'})

    results = {
        'alpha': model.params['const'],
        'beta': model.params[market_factor],
        'p_alpha': model.pvalues['const'],
        'p_beta': model.pvalues[market_factor],
        'r2': model.rsquared,
        'adj_r2': model.rsquared_adj,
        'f_stat': model.fvalue,
        'p_f': model.f_pvalue,
        'fitted': model.fittedvalues,
        'residuals': model.resid,
        'model': model,
    }
    
    return results


def plot_capm_visualizations(data, portfolio_col, results: dict, market_factor: str = 'MKT-RF'):
    x = data[market_factor]
    y = data[portfolio_col]
    
    alpha_hat = results['alpha']
    beta_hat = results['beta']
    fitted = results['fitted']
    residuals = results['residuals']
    capm_model = results['model']

    fig = make_subplots(
        rows=4, cols=2,
        specs=[
            [{"colspan": 2}, None],  
            [{}, {}],                
            [{}, {}],                
            [{"colspan": 2}, None]  
        ],
        subplot_titles=(
            f'1. CAPM: {portfolio_col} vs {market_factor}',
            '2. Residuals vs Fitted Values',
            '3. Residuals Distribution',
            '4. Coefficients Estimation (95% CI)',
            '5. Observed vs Predicted',
            '6. Time Series: Observed vs Predicted'
        ),
        vertical_spacing=0.08,
        horizontal_spacing=0.1
    )

    x_line = np.linspace(x.min(), x.max(), 100)
    y_line = alpha_hat + beta_hat * x_line
    fig.add_trace(go.Scatter(x=x, y=y, mode='markers', marker=dict(size=5, color='royalblue', opacity=0.7), name='Observations', showlegend=False), row=1, col=1)
    fig.add_trace(go.Scatter(x=x_line, y=y_line, mode='lines', line=dict(color='red', width=2), name='CAPM Line', showlegend=False), row=1, col=1)

    fig.add_trace(go.Scatter(x=fitted, y=residuals, mode='markers', marker=dict(size=5, color='orange', opacity=0.7), name='Residuals', showlegend=False), row=2, col=1)
    fig.add_hline(y=0, line_color='white', line_dash='dash', row=2, col=1)
    
    fig.add_trace(go.Histogram(x=residuals, nbinsx=25, marker=dict(color='magenta', opacity=0.7), name='Distribution', showlegend=False), row=2, col=2)

    coef_df = capm_model.conf_int().rename(columns={0:'lower', 1:'upper'})
    coef_df.index = ['Alpha', 'Beta']
    coef_df['estimate'] = capm_model.params.values

    for param, color in zip(['Beta', 'Alpha'], ['gold', 'cyan']):
        est = coef_df.loc[param, 'estimate']
        lower = coef_df.loc[param, 'lower']
        upper = coef_df.loc[param, 'upper']
        
        fig.add_trace(go.Scatter(
            x=[est], 
            y=[param],
            mode='markers+text',
            marker=dict(symbol='square', size=10, color=color),
            error_x=dict(
                type='data', 
                symmetric=False,
                array=[upper - est],
                arrayminus=[est - lower],
                color=color, 
                thickness=2, 
                width=8
            ),
            text=[f"{est:.4f} [95% CI: {lower:.4f}, {upper:.4f}]"],
            textposition="top center",
            textfont=dict(color='white', size=11),
            showlegend=False
        ), row=3, col=1)

    # --- 5) Observed vs Predicted (Row 3, Col 2) ---
    fig.add_trace(go.Scatter(x=y, y=fitted, mode='markers', marker=dict(size=5, color='lightpink', opacity=0.7), name='Obs vs Pred', showlegend=False), row=3, col=2)
    fig.add_trace(go.Scatter(x=[y.min(), y.max()], y=[y.min(), y.max()], mode='lines', line=dict(color='white', width=1.5, dash='dash'), name='y=x', showlegend=False), row=3, col=2)

    # --- 6) Time Series (Row 4, Col 1 - Colspan 2) ---
    fig.add_trace(go.Scatter(x=data.index, y=y, mode='lines', line=dict(color='royalblue', width=1.5), name='Observed', showlegend=False), row=4, col=1)
    fig.add_trace(go.Scatter(x=data.index, y=fitted, mode='lines', line=dict(color='red', width=1.5), name='Predicted', showlegend=False), row=4, col=1)

    fig.update_layout(
        title=f"CAPM Dashboard: {portfolio_col}",
        template='plotly_dark', 
        height=1200, 
        width=1000,
        hovermode="x unified",
        showlegend=False,
        margin=dict(l=50, r=50, t=80, b=50) 
    )
    
    fig.update_xaxes(title_text=f"{market_factor} (%)", row=1, col=1)
    fig.update_yaxes(title_text=f"{portfolio_col} (%)", row=1, col=1)
    
    fig.update_xaxes(title_text="Fitted Values", row=2, col=1)
    fig.update_yaxes(title_text="Residuals", row=2, col=1)
    
    fig.update_xaxes(title_text="Estimated Value", row=3, col=1, showgrid=True, gridcolor='rgba(255,255,255,0.1)')
    fig.update_yaxes(showgrid=True, gridcolor='rgba(255,255,255,0.1)', row=3, col=1)
    
    fig.update_xaxes(title_text="Observed Returns", row=3, col=2)
    fig.update_yaxes(title_text="Predicted Returns", row=3, col=2)
    
    fig.show()

capm_results = run_capm_analysis(df_final, selected_portfolio)
plot_capm_visualizations(df_final, selected_portfolio, capm_results)

### Reusable helper functions for the multi-factor models

**What the code does.** This cell defines the toolkit reused by every multi-factor regression from this point on (FF3, Carhart, FF5), so the logic is written once instead of being duplicated per model:

- `run_multifactor_analysis(data, portfolio_col, factors)` — generalises `run_capm_analysis` to any number of risk factors: fits `statsmodels.OLS` on `[const] + factors` and returns coefficients, $p$-values, confidence intervals, $R^2$/adjusted $R^2$, the $F$-statistic, fitted values, residuals, and the fitted model object itself (needed for nested tests).
- `compare_models_f_test(restricted_model, unrestricted_model)` — wraps `statsmodels.stats.anova_lm` to perform a **nested F-test**: given a smaller ("restricted") model and a larger ("unrestricted") model that contains it (e.g. CAPM ⊂ FF3 ⊂ Carhart, and FF3 ⊂ FF5), it tests $H_0$: the extra factors are jointly zero. This is the formal statistical answer to the notebook's central question — do the added factors genuinely help, or is any R² increase just overfitting noise?
- `plot_scatter_matrix`, `plot_multifactor_dashboard`, `plot_model_comparison_charts`, `plot_all_models_comparison` — the plotting counterparts: a pairwise scatter matrix of the portfolio against its factors, a 3-panel dashboard (observed vs. predicted, coefficient forest plot, residual histogram) for a single multi-factor model, and grouped bar charts comparing alpha/beta/R²/adjusted R² either between two models or across all four models at once.

**Why this matters on the ML side.** The nested F-test is the correct way to compare linear models of different complexity that are subsets of one another — it is exactly the special case of ANOVA used for model selection in linear regression, and it is more rigorous here than simply comparing raw $R^2$ values, since $R^2$ mechanically increases (or stays flat) whenever a variable is added, even a pure noise variable. This cell produces no output on its own; it only registers the functions used by every following section.

In [8]:
def plot_scatter_matrix(data, portfolio_col, factors, model_name):
 
    model_vars = [portfolio_col] + factors
    cols_to_plot = [col for col in model_vars if col in data.columns]
    
    if len(cols_to_plot) < 2:
        print(f"Error: Missing factors for {model_name} model in the data.")
        return
    
    fig = px.scatter_matrix(
        data,
        dimensions=cols_to_plot,
        title=f"{model_name} Scatter Matrix: {portfolio_col} vs Factors",
        template='plotly_dark',
        width=1000,
        height=1000
    )

    fig.update_traces(
        marker=dict(size=4, color='royalblue', opacity=0.6),
        diagonal_visible=False
    )

    fig.update_layout(
        hovermode='closest',
        margin=dict(l=50, r=50, t=80, b=50)
    )

    fig.show()
    
def run_multifactor_analysis(data, portfolio_col, factors):
    
    y = data[portfolio_col]
    X = sm.add_constant(data[factors])

    model = sm.OLS(y, X).fit()

    results = {
        'params': model.params,
        'pvalues': model.pvalues,
        'conf_int': model.conf_int(),
        'r2': model.rsquared,
        'adj_r2': model.rsquared_adj,
        'f_stat': model.fvalue,
        'p_f': model.f_pvalue,
        'fitted': model.fittedvalues,
        'residuals': model.resid,
        'model': model,
    }
    
    return results

def compare_models_f_test(restricted_model, unrestricted_model):
    return sm.stats.anova_lm(restricted_model, unrestricted_model)

def plot_multifactor_dashboard(data, portfolio_col, results, factors):
    
    y = data[portfolio_col]
    fitted = results['fitted']
    residuals = results['residuals']
    
    fig = make_subplots(
        rows=2, cols=2,
        specs=[
            [{"colspan": 2}, None],
            [{}, {}]
        ],
        subplot_titles=(
            f'1. Observed vs. Predicted - {len(factors)}-Factor Model',
            '2. Coefficient Estimates (95% CI)',
            '3. Residuals Distribution'
        ),
        vertical_spacing=0.15,
        horizontal_spacing=0.1
    )

    fig.add_trace(go.Scatter(x=y, y=fitted, mode='markers', marker=dict(size=6, color='royalblue', opacity=0.7), name='Obs vs Predicted'), row=1, col=1)
    fig.add_trace(go.Scatter(x=[y.min(), y.max()], y=[y.min(), y.max()], mode='lines', line=dict(color='white', width=1.5, dash='dash'), name='y=x'), row=1, col=1)
    fig.update_xaxes(title_text="Observed Returns (%)", row=1, col=1)
    fig.update_yaxes(title_text="Predicted Returns (%)", row=1, col=1)

    coef_df = results['conf_int'].rename(columns={0:'lower', 1:'upper'})
    coef_df['estimate'] = results['params']
    coef_df.index = ['Alpha'] + factors
    
    colors = ['cyan', 'gold', 'lime', 'orange', 'magenta', 'yellow', 'pink']
    
    for i, param in enumerate(coef_df.index):
        est = coef_df.loc[param, 'estimate']
        lower = coef_df.loc[param, 'lower']
        upper = coef_df.loc[param, 'upper']
        
        fig.add_trace(go.Scatter(
            x=[est], y=[param], mode='markers',
            marker=dict(symbol='square', size=10, color=colors[i % len(colors)]),
            error_x=dict(type='data', symmetric=False, array=[upper - est], arrayminus=[est - lower], color=colors[i % len(colors)], thickness=2, width=8),
            name=param
        ), row=2, col=1)
    fig.update_xaxes(title_text="Estimated Value", row=2, col=1, zeroline=True, zerolinewidth=1, zerolinecolor='grey')
    fig.update_yaxes(categoryorder='array', categoryarray=list(reversed(coef_df.index)), row=2, col=1)

    fig.add_trace(go.Histogram(x=residuals, nbinsx=25, marker=dict(color='magenta', opacity=0.7), name='Distribution'), row=2, col=2)
    fig.update_xaxes(title_text="Residuals", row=2, col=2)
    fig.update_yaxes(title_text="Frequency", row=2, col=2)

    fig.update_layout(
        title=f"Factor Model Dashboard: {portfolio_col}",
        template='plotly_dark', height=800, width=1000, showlegend=False
    )
    fig.show()

def plot_model_comparison_charts(summary_df, model1_name, model2_name):
    
    alpha_col1, beta_col1, r2_col1, adj_r2_col1 = f'Alpha_{model1_name}', f'Beta_{model1_name}', f'R2_{model1_name}', f'Adj_R2_{model1_name}'
    alpha_col2, beta_col2, r2_col2, adj_r2_col2 = f'Alpha_{model2_name}', f'Beta_{model2_name}', f'R2_{model2_name}', f'Adj_R2_{model2_name}'
    
    fig = make_subplots(
        rows=2, cols=2,
        subplot_titles=(
            f"Alpha Comparison ({model1_name} vs {model2_name})",
            f"Market Beta Comparison ({model1_name} vs {model2_name})",
            "R-squared Comparison",
            "Adjusted R-squared Comparison"
        ),
        vertical_spacing=0.2,
        horizontal_spacing=0.1
    )
    
    fig.add_trace(go.Bar(name=f'Alpha {model1_name}', x=summary_df['Portfolio'], y=summary_df[alpha_col1], marker_color='cyan'), row=1, col=1)
    fig.add_trace(go.Bar(name=f'Alpha {model2_name}', x=summary_df['Portfolio'], y=summary_df[alpha_col2], marker_color='lime'), row=1, col=1)
    
    fig.add_trace(go.Bar(name=f'Beta {model1_name}', x=summary_df['Portfolio'], y=summary_df[beta_col1], marker_color='cyan', showlegend=False), row=1, col=2)
    fig.add_trace(go.Bar(name=f'Beta {model2_name}', x=summary_df['Portfolio'], y=summary_df[beta_col2], marker_color='lime', showlegend=False), row=1, col=2)

    fig.add_trace(go.Bar(name=f'R-squared {model1_name}', x=summary_df['Portfolio'], y=summary_df[r2_col1], marker_color='cyan', showlegend=False), row=2, col=1)
    fig.add_trace(go.Bar(name=f'R-squared {model2_name}', x=summary_df['Portfolio'], y=summary_df[r2_col2], marker_color='lime', showlegend=False), row=2, col=1)

    fig.add_trace(go.Bar(name=f'Adj. R-squared {model1_name}', x=summary_df['Portfolio'], y=summary_df[adj_r2_col1], marker_color='cyan', showlegend=False), row=2, col=2)
    fig.add_trace(go.Bar(name=f'Adj. R-squared {model2_name}', x=summary_df['Portfolio'], y=summary_df[adj_r2_col2], marker_color='lime', showlegend=False), row=2, col=2)

    fig.update_layout(
        title_text=f"Model Comparison: {model1_name} vs. {model2_name}",
        template='plotly_dark', height=800, barmode='group',
        legend_title_text='Model'
    )
    fig.show()

def plot_all_models_comparison(summary_df):

    models = ['CAPM', 'FF3', 'Carhart', 'FF5']
    colors = ['cyan', 'lime', 'gold', 'magenta']

    fig = make_subplots(
        rows=2, cols=2,
        subplot_titles=(
            "Alpha Comparison (All Models)",
            "Market Beta Comparison (All Models)",
            "R-squared Comparison",
            "Adjusted R-squared Comparison"
        ),
        vertical_spacing=0.2,
        horizontal_spacing=0.1
    )

    for i, model in enumerate(models):
        fig.add_trace(go.Bar(name=f'{model}', x=summary_df['Portfolio'], y=summary_df[f'Alpha_{model}'], marker_color=colors[i]), row=1, col=1)

    metrics = [('Beta', 1, 2), ('R2', 2, 1), ('Adj_R2', 2, 2)]
    for metric, r, c in metrics:
        for i, model in enumerate(models):
            fig.add_trace(go.Bar(name=f'{model}', x=summary_df['Portfolio'], y=summary_df[f'{metric}_{model}'], marker_color=colors[i], showlegend=False), row=r, col=c)

    fig.update_layout(
        title_text="Global Model Comparison: CAPM vs FF3 vs Carhart vs FF5",
        template='plotly_dark', height=900, barmode='group',
        legend_title_text='Model'
    )
    fig.show()

### Factor sets for each model

In [9]:
ff3_factors = ['MKT-RF', 'SMB', 'HML']
carhart_factors = ['MKT-RF', 'SMB', 'HML', 'MOM']
ff5_factors = ['MKT-RF', 'SMB', 'HML', 'RMW', 'CMA']

In [10]:
comparison_results = []

for port in portfolio_to_plot:
    portfolio_col = f'{port}_Excess'

    # Recalcul des 4 modèles pour CE portefeuille spécifiquement
    capm_res = run_capm_analysis(df_final, portfolio_col, market_factor='MKT-RF')
    ff3_res = run_multifactor_analysis(df_final, portfolio_col, ff3_factors)
    carhart_res = run_multifactor_analysis(df_final, portfolio_col, carhart_factors)
    ff5_res = run_multifactor_analysis(df_final, portfolio_col, ff5_factors)

    # Tests F emboîtés propres à ce portefeuille
    f_test_capm_ff3 = compare_models_f_test(capm_res['model'], ff3_res['model'])
    p_val_nested = f_test_capm_ff3['Pr(>F)'][1]

    f_test_ff3_carhart = compare_models_f_test(ff3_res['model'], carhart_res['model'])
    p_val_carhart = f_test_ff3_carhart['Pr(>F)'][1]

    f_test_ff3_ff5 = compare_models_f_test(ff3_res['model'], ff5_res['model'])
    p_val_ff5 = f_test_ff3_ff5['Pr(>F)'][1]

    comparison_results.append({
        'Portfolio': port,
        'Alpha_CAPM': capm_res['alpha'],
        'Beta_CAPM': capm_res['beta'],
        'R2_CAPM': capm_res['r2'],
        'Adj_R2_CAPM': capm_res['adj_r2'],
        'Alpha_FF3': ff3_res['params']['const'],
        'Beta_FF3': ff3_res['params']['MKT-RF'],
        'R2_FF3': ff3_res['r2'],
        'Adj_R2_FF3': ff3_res['adj_r2'],
        'F_test_p_value_CAPM_FF3': p_val_nested,
        'Alpha_Carhart': carhart_res['params']['const'],
        'Beta_Carhart': carhart_res['params']['MKT-RF'],
        'R2_Carhart': carhart_res['r2'],
        'Adj_R2_Carhart': carhart_res['adj_r2'],
        'F_test_p_value_FF3_Carhart': p_val_carhart,
        'Alpha_FF5': ff5_res['params']['const'],
        'Beta_FF5': ff5_res['params']['MKT-RF'],
        'R2_FF5': ff5_res['r2'],
        'Adj_R2_FF5': ff5_res['adj_r2'],
        'F_test_p_value_FF3_FF5': p_val_ff5,
    })

comparison_df = pd.DataFrame(comparison_results)

### 2. Fama-French Three-Factor Model, portfolio by portfolio

**What the code does.** For each portfolio, this cell re-estimates CAPM and FF3 side by side, runs the nested F-test (`H₀: SMB = HML = 0`), prints the joint-significance verdict, and — for the portfolio currently selected (`Total_Portfolio`) — prints the full FF3 coefficient table and displays the scatter matrix + multi-factor diagnostic dashboard. It closes with a grouped bar chart comparing CAPM vs FF3 across all portfolios (alpha, beta, R², adjusted R²), built from `comparison_df`.

**Results obtained.**

| Portfolio | SMB=HML=0 F-stat | p-value | Verdict |
|---|---|---|---|
| Growth | 42.42 | <0.0001 | reject H₀ |
| Value | 73.45 | <0.0001 | reject H₀ |
| Small_Mid | 35.23 | <0.0001 | reject H₀ |
| Defensif | 10.93 | <0.0001 | reject H₀ |
| Qualite | 1.08 | 0.342 | **fail to reject** |
| Cyclique | 9.98 | <0.0001 | reject H₀ |
| Total_Portfolio | 20.18 | <0.0001 | reject H₀ |

For `Total_Portfolio`: α = 0.288 (p = 0.013), β(MKT-RF) = 0.957, β(SMB) = 0.014 (p = 0.741, n.s.), β(HML) = 0.180 (p < 0.001), R² = 0.929 (up from 0.905 under CAPM).

**Economic interpretation.** Adding SMB and HML is statistically significant for 6 of the 7 portfolios — the clear exception is `Qualite`, whose CAPM fit was already reasonably explained by the market alone and which was built from large, established names (JNJ, COST, ADP, MSCI) with limited size/value tilt. At the aggregate level, the positive and significant HML loading (0.18) is notable given the portfolio is dominated by mega-cap growth names (via the Growth sub-portfolio): it likely reflects the value tilt embedded in the Value, Defensif and Cyclique sleeves rather than the total portfolio being "value" per se — SMB, in contrast, is statistically indistinguishable from zero for the aggregate, consistent with a portfolio built mostly from large caps where small-cap exposure roughly nets out.

In [11]:
for port in portfolio_to_plot:
    portfolio_col = f'{port}_Excess'
    print(f"\n--- Analysis for portfolio: {port} ---")

    capm_results = run_capm_analysis(df_final, portfolio_col, market_factor='MKT-RF')
    ff3_results = run_multifactor_analysis(df_final, portfolio_col, ff3_factors)
    
    f_test_result = compare_models_f_test(capm_results['model'], ff3_results['model'])
    f_stat_nested = f_test_result['F'][1]
    p_val_nested = f_test_result['Pr(>F)'][1]
    
    print("\n--- Joint Nullity Test for Style Factors (SMB=HML=0) ---")
    print(f"F-statistic (nested test): {f_stat_nested:.4f}")
    print(f"P-value (nested test)    : {p_val_nested:.6f}")
    if p_val_nested < 0.05:
        print("=> Conclusion: Reject H0. Adding SMB and HML is statistically significant.")
    else:
        print("=> Conclusion: Do not reject H0. Adding SMB and HML does not significantly improve the model over CAPM.")
        
    if port == selected_portfolio:
            print(f"\n--- Fama-French 3 Results for {port} ---")
            ff3_summary = pd.DataFrame({
                'Coefficient': ff3_results['params'],
                'Std Error': ff3_results['model'].bse,
                't-stat': ff3_results['model'].tvalues,
                'p-value': ff3_results['pvalues']
            }).round(4)
            ff3_summary.index = ['Alpha'] + ff3_factors
            print(ff3_summary)
            print("-" * 50)
            print(f"R-squared         : {ff3_results['r2']:.4f}")
            print(f"Adjusted R-squared: {ff3_results['adj_r2']:.4f}")
            print(f"F-stat (global)   : {ff3_results['f_stat']:.4f} (p-value: {ff3_results['p_f']:.6f})")
            
            # Affichage des dashboards et scatter matrix
            plot_scatter_matrix(df_final, portfolio_col, ff3_factors, "FF3")
            plot_multifactor_dashboard(df_final, portfolio_col, ff3_results, ff3_factors)
            
print("\n\n" + "="*25 + " COMPARATIVE PLOTS " + "="*25)
plot_model_comparison_charts(comparison_df, 'CAPM', 'FF3')


--- Analysis for portfolio: Growth ---

--- Joint Nullity Test for Style Factors (SMB=HML=0) ---
F-statistic (nested test): 42.4209
P-value (nested test)    : 0.000000
=> Conclusion: Reject H0. Adding SMB and HML is statistically significant.

--- Analysis for portfolio: Value ---

--- Joint Nullity Test for Style Factors (SMB=HML=0) ---
F-statistic (nested test): 73.4478
P-value (nested test)    : 0.000000
=> Conclusion: Reject H0. Adding SMB and HML is statistically significant.

--- Analysis for portfolio: Small_Mid ---

--- Joint Nullity Test for Style Factors (SMB=HML=0) ---
F-statistic (nested test): 35.2253
P-value (nested test)    : 0.000000
=> Conclusion: Reject H0. Adding SMB and HML is statistically significant.

--- Analysis for portfolio: Defensif ---

--- Joint Nullity Test for Style Factors (SMB=HML=0) ---
F-statistic (nested test): 10.9298
P-value (nested test)    : 0.000043
=> Conclusion: Reject H0. Adding SMB and HML is statistically significant.

--- Analysis for po



========================= COMPARATIVE PLOTS =========================


### 3. Carhart Four-Factor Model, portfolio by portfolio

**What the code does.** Same structure as the FF3 section, but the nested test now compares FF3 vs. Carhart (`H₀: MOM = 0`), i.e. it asks whether momentum adds explanatory power on top of market, size and value. The dashboard and full coefficient table are shown for `Total_Portfolio`, and the section ends with a comparison chart, this time FF3 vs. Carhart.

**Results obtained.**

| Portfolio | MOM=0 F-stat | p-value | Verdict |
|---|---|---|---|
| Growth | 2.08 | 0.152 | fail to reject |
| Value | 5.67 | 0.019 | reject H₀ |
| Small_Mid | 6.14 | 0.015 | reject H₀ |
| Defensif | 0.28 | 0.599 | fail to reject |
| Qualite | 0.02 | 0.876 | fail to reject |
| Cyclique | 0.16 | 0.687 | fail to reject |
| Total_Portfolio | 8.12 | 0.005 | reject H₀ |

For `Total_Portfolio`: α = 0.326 (p = 0.004), β(MKT-RF) = 0.938, β(SMB) = -0.019 (n.s.), β(HML) = 0.164 (p < 0.001), β(MOM) = **-0.090** (p = 0.005), R² = 0.933 (marginal gain over FF3's 0.929).

**Economic interpretation.** Momentum is significant for only 3 of 7 portfolios (Value, Small_Mid, Total_Portfolio), and where it is significant, its sign is systematically **negative** — the opposite of the "winners keep winning" effect Carhart (1997) documented in mutual fund performance. A negative MOM loading here more plausibly reflects a *contrarian/mean-reverting* tilt embedded in the value-heavy sleeves of this specific portfolio (post-2016 US equities saw momentum crash sharply around several turning points, e.g. the 2020 growth-to-value rotation and the 2022 tightening cycle) rather than a genuine, stable momentum premium. Combined with the fact that R² barely moves from FF3 to Carhart for most portfolios (e.g. Growth: 0.782 → 0.785; Qualite: 0.607 → 0.607), this is a good illustration of the notebook's opening question: momentum, in this sample, mostly looks like statistical noise or a sample-specific artifact rather than a robust additional risk factor.

In [12]:
for port in portfolio_to_plot:
    portfolio_col = f'{port}_Excess'
    print(f"\n--- Analysis for portfolio: {port} ---")
    
    ff3_results = run_multifactor_analysis(df_final, portfolio_col, ff3_factors)    
    carhart_results = run_multifactor_analysis(df_final, portfolio_col, carhart_factors)
    
    f_test_carhart = compare_models_f_test(ff3_results['model'], carhart_results['model'])
    f_stat_carhart = f_test_carhart['F'][1]
    p_val_carhart = f_test_carhart['Pr(>F)'][1]
    
    print("\n--- Nullity Test for Momentum Factor (MOM=0) ---")
    print(f"F-statistic (nested test): {f_stat_carhart:.4f}")
    print(f"P-value (nested test)    : {p_val_carhart:.6f}")
    if p_val_carhart < 0.05:
        print("=> Conclusion: Reject H0. Adding MOM is statistically significant.")
    else:
        print("=> Conclusion: Do not reject H0. Adding MOM does not significantly improve the model over FF3.")
        
    if port == selected_portfolio:
        print(f"\n--- Carhart Results for {port} ---")
        carhart_summary = pd.DataFrame({
            'Coefficient': carhart_results['params'],
            'Std Error': carhart_results['model'].bse,
            't-stat': carhart_results['model'].tvalues,
            'p-value': carhart_results['pvalues']
        }).round(4)
        carhart_summary.index = ['Alpha'] + carhart_factors
        print(carhart_summary)
        print("-" * 50)
        print(f"R-squared         : {carhart_results['r2']:.4f}")
        print(f"Adjusted R-squared: {carhart_results['adj_r2']:.4f}")
        print(f"F-stat (global)   : {carhart_results['f_stat']:.4f} (p-value: {carhart_results['p_f']:.6f})")
        

        plot_scatter_matrix(df_final, portfolio_col, carhart_factors, "Carhart-4")
        plot_multifactor_dashboard(df_final, portfolio_col, carhart_results, carhart_factors)
        
print("\n\n" + "="*25 + " COMPARATIVE PLOTS " + "="*25)
plot_model_comparison_charts(comparison_df, 'FF3', 'Carhart')




--- Analysis for portfolio: Growth ---

--- Nullity Test for Momentum Factor (MOM=0) ---
F-statistic (nested test): 2.0818
P-value (nested test)    : 0.151668
=> Conclusion: Do not reject H0. Adding MOM does not significantly improve the model over FF3.

--- Analysis for portfolio: Value ---

--- Nullity Test for Momentum Factor (MOM=0) ---
F-statistic (nested test): 5.6713
P-value (nested test)    : 0.018818
=> Conclusion: Reject H0. Adding MOM is statistically significant.

--- Analysis for portfolio: Small_Mid ---

--- Nullity Test for Momentum Factor (MOM=0) ---
F-statistic (nested test): 6.1410
P-value (nested test)    : 0.014599
=> Conclusion: Reject H0. Adding MOM is statistically significant.

--- Analysis for portfolio: Defensif ---

--- Nullity Test for Momentum Factor (MOM=0) ---
F-statistic (nested test): 0.2781
P-value (nested test)    : 0.598938
=> Conclusion: Do not reject H0. Adding MOM does not significantly improve the model over FF3.

--- Analysis for portfolio: Qua



========================= COMPARATIVE PLOTS =========================


### 4. Fama-French Five-Factor Model, portfolio by portfolio

**What the code does.** Same pattern again: FF3 vs. FF5 is tested (`H₀: RMW = CMA = 0`), the full coefficient table and dashboard are shown for `Total_Portfolio`, and the section closes with an FF3-vs-FF5 comparison chart.

**Results obtained.**

| Portfolio | p-value (RMW=CMA=0) | Verdict | R² FF3 → FF5 |
|---|---|---|---|
| Growth | 0.0027 | reject H₀ | 0.782 → 0.802 |
| Value | 0.8393 | fail to reject | 0.787 → 0.788 |
| Small_Mid | 0.3080 | fail to reject | 0.715 → 0.720 |
| Defensif | <0.0001 | reject H₀ | 0.327 → 0.437 |
| Qualite | 0.0024 | reject H₀ | 0.607 → 0.645 |
| Cyclique | 0.0930 | fail to reject (10%) | 0.599 → 0.614 |
| Total_Portfolio | 0.0001 | reject H₀ | 0.929 → 0.939 |

For `Total_Portfolio`: α = 0.281 (p = 0.010), β(MKT-RF) = 0.946, β(SMB) = 0.083 (p = 0.053, borderline), β(HML) = 0.147 (p < 0.001), β(RMW) = **0.215** (p < 0.001), β(CMA) = -0.012 (n.s.), R² = 0.939, the best fit of all four models on this portfolio.

**Economic interpretation.** Profitability and investment jointly matter for 4 of 7 portfolios, and the effect is largest exactly where intuition would suggest: `Defensif` sees its R² jump from 0.327 to 0.437 (the single largest FF5 improvement of any portfolio), consistent with staples/utilities/healthcare names (PG, KO, WMT, DUK, SO) being classic **robust-profitability, low-investment ("quality")** stocks that RMW is specifically designed to capture — a story CAPM and even FF3 were structurally unable to tell, since neither model has a profitability factor. `CMA` (investment) is never significant for `Total_Portfolio` and rarely elsewhere, suggesting that in this particular universe and sample period, profitability carries essentially all of the incremental information in the FF5 extension, while the investment factor is largely redundant — a useful, and fairly standard, empirical finding once size/value/profitability are already controlled for.

In [13]:
for port in portfolio_to_plot:
    portfolio_col = f'{port}_Excess'
    print(f"\n--- Analysis for portfolio: {port} ---")
    
    ff3_results = run_multifactor_analysis(df_final, portfolio_col, ff3_factors)    
    ff5_results = run_multifactor_analysis(df_final, portfolio_col, ff5_factors)
    
    f_test_ff5 = compare_models_f_test(ff3_results['model'], ff5_results['model'])
    f_stat_ff5 = f_test_ff5['F'][1]
    p_val_ff5 = f_test_ff5['Pr(>F)'][1]
    
    print("\n--- Nullity Test for Momentum Factor (MOM=0) ---")
    print(f"F-statistic (nested test): {f_stat_ff5:.4f}")
    print(f"P-value (nested test)    : {p_val_ff5:.6f}")
    if p_val_ff5 < 0.05:
        print("=> Conclusion: Reject H0. Adding MOM is statistically significant.")
    else:
        print("=> Conclusion: Do not reject H0. Adding MOM does not significantly improve the model over FF3.")
        
    if port == selected_portfolio:
        print(f"\n--- FF5 Results for {port} ---")
        ff5_summary = pd.DataFrame({
            'Coefficient': ff5_results['params'],
            'Std Error': ff5_results['model'].bse,
            't-stat': ff5_results['model'].tvalues,
            'p-value': ff5_results['pvalues']
        }).round(4)
        ff5_summary.index = ['Alpha'] + ff5_factors
        print(ff5_summary)
        print("-" * 50)
        print(f"R-squared         : {ff5_results['r2']:.4f}")
        print(f"Adjusted R-squared: {ff5_results['adj_r2']:.4f}")
        print(f"F-stat (global)   : {ff5_results['f_stat']:.4f} (p-value: {ff5_results['p_f']:.6f})")
        

        plot_scatter_matrix(df_final, portfolio_col, ff5_factors, "FF5-5")
        plot_multifactor_dashboard(df_final, portfolio_col, ff5_results, ff5_factors)

plot_model_comparison_charts(comparison_df, 'FF3', 'FF5')




--- Analysis for portfolio: Growth ---

--- Nullity Test for Momentum Factor (MOM=0) ---
F-statistic (nested test): 6.2322
P-value (nested test)    : 0.002667
=> Conclusion: Reject H0. Adding MOM is statistically significant.

--- Analysis for portfolio: Value ---

--- Nullity Test for Momentum Factor (MOM=0) ---
F-statistic (nested test): 0.1754
P-value (nested test)    : 0.839302
=> Conclusion: Do not reject H0. Adding MOM does not significantly improve the model over FF3.

--- Analysis for portfolio: Small_Mid ---

--- Nullity Test for Momentum Factor (MOM=0) ---
F-statistic (nested test): 1.1895
P-value (nested test)    : 0.307954
=> Conclusion: Do not reject H0. Adding MOM does not significantly improve the model over FF3.

--- Analysis for portfolio: Defensif ---

--- Nullity Test for Momentum Factor (MOM=0) ---
F-statistic (nested test): 11.6139
P-value (nested test)    : 0.000025
=> Conclusion: Reject H0. Adding MOM is statistically significant.

--- Analysis for portfolio: Qu

### Global Model Comparison: CAPM vs FF3 vs Carhart vs FF5

**What the code does.** This cell renders `plot_all_models_comparison(comparison_df)` — a single 4-panel grouped bar chart showing alpha, beta, R² and adjusted R² for all four models across all 7 portfolios simultaneously — followed by a full numeric printout of `comparison_df`, the master table built earlier in "Cross-model comparison across portfolios".

**Results obtained — in-sample R² by model:**

| Portfolio | CAPM | FF3 | Carhart | FF5 |
|---|---|---|---|---|
| Growth | 0.629 | 0.782 | 0.785 | **0.802** |
| Value | 0.529 | 0.787 | **0.797** | 0.788 |
| Small_Mid | 0.548 | 0.715 | **0.729** | 0.720 |
| Defensif | 0.205 | 0.327 | 0.328 | **0.437** |
| Qualite | 0.600 | 0.607 | 0.607 | **0.645** |
| Cyclique | 0.533 | 0.599 | 0.599 | **0.614** |
| Total_Portfolio | 0.905 | 0.929 | 0.933 | **0.939** |

Adjusted R² (which penalizes added regressors) tells essentially the same story: the FF5 adjusted R² for `Total_Portfolio` (0.937) is barely below its raw R² (0.939), confirming the extra factors are pulling real weight rather than just fitting noise; the same holds, to varying degrees, for every portfolio.

**Economic interpretation — answering the notebook's opening question.** Across all 7 portfolios, three consistent patterns emerge:

1. **The jump from CAPM to FF3 is always the largest single improvement** — market beta alone leaves a lot on the table for every style except the already highly diversified `Total_Portfolio`. Size and value are the most broadly useful factors in this sample.
2. **Momentum (Carhart) is the weakest and least consistent addition** — significant (and negative) for only 3 portfolios, with tiny R² gains almost everywhere else. Of the four models, Carhart is the one where "added complexity ≈ noise" is the most defensible reading.
3. **Profitability/investment (FF5) meaningfully re-earns its place for specific styles** — `Defensif` and `Qualite` are exactly the sub-portfolios built around stable, profitable, low-capex businesses, and they are exactly where FF5 adds the most explanatory power, a reassuring sanity check that the factor is doing what its construction (robust-minus-weak profitability) says it should.

Put together, the answer to "do richer models capture genuine risk structure, or do they simply overfit?" is **factor-dependent**: SMB/HML and, more selectively, RMW clearly carry real information; MOM, at least over 2016–2026 for this specific universe, does not clear that bar convincingly.

In [14]:
print("\n\n" + "="*25 + " MODEL COMPARISON SUMMARY " + "="*25)
plot_all_models_comparison(comparison_df)
display(comparison_df.round(4))



========================= MODEL COMPARISON SUMMARY =========================


,Portfolio,Alpha_CAPM,Beta_CAPM,R2_CAPM,Adj_R2_CAPM,Alpha_FF3,Beta_FF3,R2_FF3,Adj_R2_FF3,F_test_p_value_CAPM_FF3,Alpha_Carhart,Beta_Carhart,R2_Carhart,Adj_R2_Carhart,F_test_p_value_FF3_Carhart,Alpha_FF5,Beta_FF5,R2_FF5,Adj_R2_FF5,F_test_p_value_FF3_FF5
0,Growth,1.0267,1.2284,0.6286,0.6256,0.9340,1.2931,0.7817,0.7763,0.0000,0.9868,1.2664,0.7854,0.7782,0.1517,0.9646,1.2280,0.8024,0.7941,0.0027
1,Value,-0.1314,0.9323,0.5285,0.5246,-0.1562,0.9357,0.7870,0.7817,0.0000,-0.0860,0.9001,0.7966,0.7899,0.0188,-0.1598,0.9451,0.7876,0.7787,0.8393
2,Small_Mid,-0.4292,1.3162,0.5484,0.5447,-0.1395,1.1364,0.7146,0.7075,0.0000,-0.0225,1.0772,0.7285,0.7194,0.0146,-0.1316,1.1010,0.7202,0.7084,0.3080
3,Defensif,0.4222,0.3794,0.2050,0.1986,0.2594,0.4739,0.3267,0.3100,0.0000,0.2409,0.4832,0.3282,0.3058,0.5989,0.2146,0.4950,0.4366,0.4130,0.0000
4,Qualite,0.4759,0.7487,0.5999,0.5967,0.4708,0.7531,0.6070,0.5972,0.3423,0.4660,0.7555,0.6071,0.5940,0.8760,0.4514,0.7414,0.6449,0.6300,0.0024
5,Cyclique,0.3660,1.0815,0.5325,0.5287,0.3746,1.0699,0.5987,0.5887,0.0001,0.3553,1.0796,0.5992,0.5858,0.6872,0.3417,1.0958,0.6144,0.5982,0.0930
6,Total_Portfolio,0.2902,0.9588,0.9051,0.9043,0.2883,0.9572,0.9288,0.9270,0.0000,0.3261,0.9381,0.9333,0.9311,0.0052,0.2805,0.9456,0.9394,0.9369,0.0001


### Rolling-Window Stability Analysis

**What the code does.** `compute_rolling_model_metrics()` re-estimates a given factor model on a rolling 24-month window (instead of once over the full 2016–2026 sample) and records the market beta and adjusted R² at each window's end date. `plot_rolling_model_metrics()` runs this for all four models (CAPM, FF3, Carhart, FF5) on the currently selected portfolio (`Total_Portfolio`) and plots the rolling beta and rolling adjusted R² together, one panel per model.

**Why this matters on the ML side.** Every regression earlier in the notebook implicitly assumes the coefficients are **constant over the full 2016–2026 sample** — a strong assumption for a 10-year window spanning very different market regimes (the 2018 volatility spike, Covid in 2020, the 2022 rate-hiking cycle). A rolling regression relaxes that assumption by re-fitting the model locally, which is the standard diagnostic for **parameter (in)stability** in time-series regression — directly following up on the "Untested temporal stability" limitation flagged in the CAPM section.

**How to read the chart.** A flat rolling-beta line means the portfolio's market sensitivity has been stable through time — supporting the full-sample estimate as a reasonable single number. Sharp swings instead indicate regime-dependent risk exposure, meaning the single full-sample beta reported earlier is really an average that can mask meaningful drift (for example, a mega-cap growth-heavy portfolio's beta plausibly rose over 2020–2023 alongside increasing concentration in a handful of tech names). Rolling adjusted R² tells the same story for overall model fit: a model whose R² degrades in certain sub-periods is one whose factor structure isn't fully time-invariant, which matters for anyone using these betas out-of-sample (e.g. for hedging or risk budgeting) rather than purely for historical explanation.

In [ ]:
def compute_rolling_model_metrics(data, portfolio_col, factors, window=24):
    required_cols = [portfolio_col] + factors
    missing = [col for col in required_cols if col not in data.columns]
    if missing:
        raise ValueError(f"Missing columns for rolling analysis: {missing}")

    rows = []
    for i in range(window - 1, len(data)):
        window_data = data.iloc[i - window + 1:i + 1]
        y = window_data[portfolio_col]
        X = sm.add_constant(window_data[factors])
        model = sm.OLS(y, X).fit()

        row = {'date': window_data.index[-1]}
        for factor in factors:
            row[f'rolling_beta_{factor}'] = model.params[factor]
        row['rolling_adj_r2'] = model.rsquared_adj
        row['rolling_r2'] = model.rsquared
        rows.append(row)

    rolling_df = pd.DataFrame(rows).set_index('date')
    return rolling_df


def plot_rolling_model_metrics(data, portfolio_col, window=24, model_specs=None):
    if model_specs is None:
        model_specs = {
            'CAPM': ['MKT-RF'],
            'FF3': ['MKT-RF', 'SMB', 'HML'],
            'Carhart': ['MKT-RF', 'SMB', 'HML', 'MOM'],
            'FF5': ['MKT-RF', 'SMB', 'HML', 'RMW', 'CMA'],
        }

    results = {}
    for model_name, factors in model_specs.items():
        rolling_df = compute_rolling_model_metrics(data, portfolio_col, factors, window=window)
        results[model_name] = rolling_df

    fig = make_subplots(
        rows=2,
        cols=2,
        specs=[
            [{"secondary_y": True}, {"secondary_y": True}],
            [{"secondary_y": True}, {"secondary_y": True}],
        ],
        subplot_titles=list(model_specs.keys()),
        vertical_spacing=0.14,
        horizontal_spacing=0.12,
    )

    positions = [(1, 1), (1, 2), (2, 1), (2, 2)]
    for (model_name, rolling_df), (row, col) in zip(results.items(), positions):
        beta_col = f'rolling_beta_MKT-RF'
        fig.add_trace(
            go.Scatter(
                x=rolling_df.index,
                y=rolling_df[beta_col],
                mode='lines',
                name=f'{model_name} beta',
                line=dict(color='cyan', width=2),
            ),
            row=row,
            col=col,
        )
        fig.add_trace(
            go.Scatter(
                x=rolling_df.index,
                y=rolling_df['rolling_adj_r2'],
                mode='lines',
                name=f'{model_name} adj R²',
                line=dict(color='gold', width=2),
            ),
            row=row,
            col=col,
            secondary_y=True,
        )

        fig.update_yaxes(title_text='Rolling Beta', row=row, col=col)
        fig.update_yaxes(title_text='Rolling Adj. R²', row=row, col=col, secondary_y=True)

    fig.update_layout(
        title_text=f'Rolling Beta and Adjusted R² for {portfolio_col}',
        template='plotly_dark',
        height=850,
        width=1200,
        legend_title_text='Metric',
    )
    fig.show()

    return results

rolling_results = plot_rolling_model_metrics(
    df_final,
    f"{selected_portfolio}_Excess",
    window=24
)